# 02 — Jonnathan: Pelatihan Model Video (M0–M5), Ablasi E1–E7, & Transfer ASEAN FSL (X1–X4)
**Sub-tim Arsitektur & Pelatihan Video**

Notebook ini mencakup:
- **M0 / M5**: `StudentPoseTransformer` (~1.3M parameter, input posisi + kecepatan $\Delta x$).
- **M1**: `PoseModel` (6-blok ST-GCN + `PhonologyMultitaskHead`).
- **M2**: `RGBOnlyModel` (VideoMAE token pooler + BiGRU).
- **M3 / M4**: `TwoStreamFusionModel` (Cross-Attention + Gated Fusion + Modality Dropout `p=0.1`).
- **Distilasi Pengetahuan (KD)**: $\mathcal{L}_{\text{KD}} = \alpha \cdot \text{CE}(y, s) + (1-\alpha) T^2 \text{KL}(\sigma(t/T) \| \sigma(s/T))$ dengan $\alpha=0.5, T=4$.

In [ ]:
import torch
from signbridge.models.stgcn import PoseModel
from signbridge.models.student import StudentPoseTransformer
from signbridge.models.fusion import TwoStreamFusionModel, RGBOnlyModel

kp = torch.randn(2, 64, 86, 3)
rgb = torch.randn(2, 8, 768)

student = StudentPoseTransformer(num_classes=32)
pose_m1 = PoseModel(num_classes=32, use_phonology=True)
rgb_m2 = RGBOnlyModel(num_classes=32)
teacher_m4 = TwoStreamFusionModel(num_classes=32, fusion_layers=2, use_phonology=True)

print('M0/M5 Student output:', student(kp).shape)
print('M1 Pose ST-GCN output:', pose_m1(kp)['logits'].shape)
print('M2 RGB-Only output:', rgb_m2(rgb)['logits'].shape)
out_m4 = teacher_m4(kp, rgb)
print('M4 Teacher Fusion output:', out_m4['logits'].shape, 'Gate mean:', float(out_m4['gate'].mean()))